# Train and validate

Reviewed human FPV baseline; no NVIDIA API calls, cloud runtime creation or robot commands. Set `CAFEROOMBA_REVISION` to the full published implementation commit SHA. Mount persistent storage and set `CAFEROOMBA_WORKSPACE` to the same location in notebooks 01–03. For consumer Colab, explicitly mount Drive if wanted: `from google.colab import drive; drive.mount('/content/drive')`. Enterprise storage/auth varies; use your persistent filesystem. Ordinary `/content` and `/tmp` are not durable across runtime deletion. Never paste keys into cells. See `docs/COLAB_TRAINING.md`.

In [ ]:
import os, re, subprocess, sys, shutil
from pathlib import Path
REPO_REVISION = os.environ.get("CAFEROOMBA_REVISION", "SET_IMPLEMENTATION_COMMIT_SHA")
REPO = Path(os.environ.get("CAFEROOMBA_REPO", "/content/caferoomba"))
if not re.fullmatch(r"[0-9a-f]{40}", REPO_REVISION):
    raise ValueError("Set CAFEROOMBA_REVISION to the full published implementation commit SHA.")
if not REPO.exists():
    subprocess.run(["git", "clone", "https://github.com/EdwinKestler/caferoomba.git", str(REPO)], check=True)
    subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", REPO_REVISION], check=True)
actual = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
if actual != REPO_REVISION:
    raise ValueError("Existing checkout differs; use a separate checkout without overwriting local work.")
if subprocess.check_output(["git", "-C", str(REPO), "status", "--porcelain"], text=True).strip():
    raise ValueError("Use a clean checkout for reproducible source provenance.")
if sys.version_info < (3, 11):
    raise RuntimeError("Python 3.11 or newer required.")
# Preserve an existing CUDA-enabled Torch installation; do not force CPU-only wheels.
subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(REPO) + "[cpu,dev]"], check=True)
sys.path.insert(0, str(REPO / "src"))
if not shutil.which("ffmpeg") or not shutil.which("ffprobe"):
    raise RuntimeError("Install ffmpeg/ffprobe before preparing videos.")
print("source revision:", actual)


In [ ]:
workspace_text = os.environ.get("CAFEROOMBA_WORKSPACE", "")
if not workspace_text:
    raise ValueError("Set CAFEROOMBA_WORKSPACE to your mounted persistent workspace.")
WORKSPACE = Path(workspace_text).resolve()
if not WORKSPACE.is_dir():
    raise ValueError("Workspace must already exist on persistent storage.")
DATASET = WORKSPACE / "dataset-v1"
RUN = WORKSPACE / "run-v1"
MANIFEST = DATASET / "manifest.json"


In [ ]:
import torch
from caferoomba.learning.pipeline import train_run
DEVICE = os.environ.get("CAFEROOMBA_TRAIN_DEVICE", "cpu")
EPOCHS = int(os.environ.get("CAFEROOMBA_EPOCHS", "10"))  # total, including resumed epochs
BATCH_SIZE = int(os.environ.get("CAFEROOMBA_BATCH_SIZE", "8"))
RESUME = os.environ.get("CAFEROOMBA_RESUME", "")  # explicit path to this run's last.pt
if DEVICE.startswith("cuda") and not torch.cuda.is_available():
    raise RuntimeError("CUDA requested but unavailable; choose an appropriate runtime or cpu.")
report = train_run(MANIFEST, RUN, epochs=EPOCHS, batch_size=BATCH_SIZE, device=DEVICE,
                   resume=Path(RESUME) if RESUME else None)
print({key: report[key] for key in ("checkpoint", "last_checkpoint", "dataset_sha256")})


Every epoch visits the training split in shuffled batches. Validation selects `best.pt`; `last.pt` stores optimizer/RNG state for explicit resume. Keep the whole dataset/run on persistent storage. The test split is not evaluated here. Finish model selection before notebook 03.